# L7 demo: features, and the leak a scaler can hide

This notebook builds a small per-unit feature set for the NASA C-MAPSS turbofan
data (FD001), then predicts Remaining Useful Life (RUL) two ways: once with a
`StandardScaler` fit on the training engines only, and once with the same
scaler fit on training **and** test data combined, the "quick, obviously
fine" shortcut that is actually the single most common leakage bug in a
feature pipeline. We quantify the gap, and, just as importantly, look at why
the gap is not always dramatic, which is the more useful lesson than a scare
number.

## Get the data

FD001 (single operating condition, single fault mode) is the smallest of the
four C-MAPSS subsets, run-to-failure trajectories for 100 simulated turbofan
engines. NASA does not publish a single stable direct-download URL for this
data (the Prognostics Center of Excellence has moved its repository more
than once), so this notebook expects the three FD001 files placed in
`.cache/CMAPSS/` rather than fetching them itself:

1. Download the "Turbofan Engine Degradation Simulation Data Set" from the
   [NASA Prognostics Center of Excellence Data Set Repository](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/).
2. Unzip it and copy `train_FD001.txt`, `test_FD001.txt`, and `RUL_FD001.txt`
   into `lectures/l07/.cache/CMAPSS/`.

This mirrors A4's instructions, since the assignment uses the same files.

In [ ]:
from pathlib import Path

CACHE = Path('.cache/CMAPSS')
TRAIN_FILE = CACHE / 'train_FD001.txt'
TEST_FILE = CACHE / 'test_FD001.txt'
RUL_FILE = CACHE / 'RUL_FD001.txt'

if not (TRAIN_FILE.exists() and TEST_FILE.exists() and RUL_FILE.exists()):
    raise FileNotFoundError(
        f'Place train_FD001.txt, test_FD001.txt, and RUL_FD001.txt in {CACHE}/ '
        'first -- see the instructions above.'
    )


## Stage 1: ingest

Each row is one engine, one cycle: a unit id, the cycle number, three
operational settings, and 21 sensor channels, whitespace separated with no
header. The test file is the same shape, except every unit's trajectory is
truncated **before** failure; `RUL_FD001.txt` holds the true remaining cycles
at that cutoff, one value per test unit, in unit order.

In [ ]:
import numpy as np
import pandas as pd

N_SETTINGS, N_SENSORS = 3, 21
COLUMNS = (
    ['unit', 'cycle']
    + [f'setting{i + 1}' for i in range(N_SETTINGS)]
    + [f'sensor{i + 1}' for i in range(N_SENSORS)]
)

def ingest(path):
    df = pd.read_csv(path, sep=r'\s+', header=None, names=COLUMNS)
    df[['unit', 'cycle']] = df[['unit', 'cycle']].astype(int)
    return df

train_raw = ingest(TRAIN_FILE)
test_raw = ingest(TEST_FILE)
rul_true = pd.read_csv(RUL_FILE, header=None, names=['rul'])
rul_true.index = np.arange(1, len(rul_true) + 1)   # unit ids are 1-indexed, in file order

print(train_raw.shape, 'train rows;', train_raw['unit'].nunique(), 'engines')
print(test_raw.shape, 'test rows;', test_raw['unit'].nunique(), 'engines')
train_raw.head(3)


## Stage 2: the RUL target

Train is run-to-failure, so an engine's RUL at any cycle is simply its final
cycle minus the current one. Most published C-MAPSS baselines **clip** this
at a ceiling (125 cycles here): an engine on cycle 3 of 300 is not
meaningfully "healthier" than one on cycle 30, since degradation only
becomes informative once it starts, and an unclipped linear target rewards a
model for memorizing each unit's total lifetime rather than reading the
degradation signal.

In [ ]:
MAX_RUL = 125

def add_rul(df, max_rul=MAX_RUL):
    df = df.copy()
    max_cycle = df.groupby('unit')['cycle'].transform('max')
    df['rul'] = (max_cycle - df['cycle']).clip(upper=max_rul)
    return df

train_raw = add_rul(train_raw)
train_raw[['unit', 'cycle', 'rul']].groupby('unit').tail(1).head(3)


## Stage 3: per-unit time-series features

Three feature families, computed **within each engine's own trajectory**
(`groupby('unit')`, never across engines): a rolling mean and standard
deviation, a delta from that engine's first recorded cycle, and the
cycle-to-cycle rate of change. All three read directly off the module's
Topics list, and all three would be silently wrong if computed before
grouping by unit, since cycle 1 of engine 12 has nothing to do with cycle 300
of engine 7.

In [ ]:
KEY_SENSORS = ['sensor2', 'sensor5', 'sensor8']   # the channels with a real degradation trend

def engineer_features(df, sensors=KEY_SENSORS, window=5):
    df = df.sort_values(['unit', 'cycle']).copy()
    g = df.groupby('unit', group_keys=False)
    for s in sensors:
        df[f'{s}_roll_mean'] = g[s].transform(lambda x: x.rolling(window, min_periods=1).mean())
        df[f'{s}_roll_std'] = g[s].transform(lambda x: x.rolling(window, min_periods=1).std().fillna(0))
        df[f'{s}_delta0'] = g[s].transform(lambda x: x - x.iloc[0])
        df[f'{s}_roc'] = g[s].transform(lambda x: x.diff().fillna(0))
    return df

train_fe = engineer_features(train_raw)
test_fe = engineer_features(test_raw)
feature_cols = [c for c in train_fe.columns if any(c.startswith(s) for s in KEY_SENSORS)]
print(len(feature_cols), 'engineered features from', len(KEY_SENSORS), 'sensor channels')


## Stage 4a: the wrong way

Fit the scaler on training **and** test rows together, "because we have the
data anyway." This is exactly the bug the module flags as the most common
one in a feature pipeline: nothing here looks unusual, and nothing here
raises an error. The scaler simply learns statistics informed by data the
model will later be judged against.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

def last_cycle_per_unit(df):
    return df.sort_values('cycle').groupby('unit').tail(1).sort_values('unit')

last_rows = last_cycle_per_unit(test_fe)
y_true = rul_true.loc[last_rows['unit'], 'rul'].to_numpy()

# The leak: .fit on the concatenation of train and test.
combined = pd.concat([train_fe[feature_cols], test_fe[feature_cols]], axis=0)
leaky_scaler = StandardScaler().fit(combined)

leaky_model = Ridge(alpha=10.0).fit(
    leaky_scaler.transform(train_fe[feature_cols]), train_fe['rul']
)
pred_leaky = leaky_model.predict(leaky_scaler.transform(last_rows[feature_cols]))
rmse_leaky = mean_squared_error(y_true, pred_leaky) ** 0.5
print(f'leaky-scaler RMSE:  {rmse_leaky:.3f} RUL-cycles')


## Stage 4b: the right way, as an `sklearn` `Pipeline`

Wrapping the scaler and the model in a single `Pipeline` does not just save
keystrokes. It makes the leak structurally harder to write by accident:
`pipeline.fit(X_train, y_train)` has no way to see `X_test`, so there is no
line of code left where "fit on everything" could sneak back in.

In [ ]:
from sklearn.pipeline import Pipeline

correct_pipeline = Pipeline([
    ('scale', StandardScaler()),
    ('model', Ridge(alpha=10.0)),
]).fit(train_fe[feature_cols], train_fe['rul'])

pred_correct = correct_pipeline.predict(last_rows[feature_cols])
rmse_correct = mean_squared_error(y_true, pred_correct) ** 0.5
print(f'correct-pipeline RMSE: {rmse_correct:.3f} RUL-cycles')
print(f'leaky-scaler RMSE:     {rmse_leaky:.3f} RUL-cycles')
print(f'gap:                   {rmse_leaky - rmse_correct:+.3f} RUL-cycles')


## Reading the gap honestly

Run this yourself and the gap will likely be small, sometimes a fraction of
a cycle, and it can even run in the "wrong" direction on a given resample.
That is a real result, not a failed demo, and it is worth understanding why.
Train and test engines here are the *same simulated fleet under the same
operating regime*, just cut off before failure, so a scaler fit on both
looks almost like a scaler fit on training alone. `Ridge` also only degrades
gracefully as its inputs are rescaled, since the penalty interacts with
scale but does not ignore it the way plain least-squares would (an
unregularized linear fit is *exactly* scale-invariant, and would show a
difference of zero, which would make this look like a broken demo rather
than a genuine one).

None of that is a reason to relax the rule. It is the reason the rule has to
be structural rather than case-by-case: you cannot know in advance whether
this month's data, this quarter's model, or next year's dataset will be the
one where the leak is large. A held-out test set whose statistics quietly
informed training is a mistake whether or not this particular run made it
obvious, and a random or `ColumnTransformer`-free script gives you no way to
even check. The much larger and easier way to leak in this exact dataset,
letting a random split scatter one engine's cycles across both train and
test, is deliberately not shown here; that is L6's neighbor, L8's leakage
taxonomy, next session.

## Persisting the fitted pipeline

The whole point of fitting inside a `Pipeline` is that the fitted object,
scaler statistics and all, is one artifact you can save, version, and reload
exactly, rather than a script you have to rerun and hope reproduces the same
numbers.

In [ ]:
import joblib

MODEL_PATH = CACHE.parent / 'rul_pipeline.joblib'
joblib.dump(correct_pipeline, MODEL_PATH)

reloaded = joblib.load(MODEL_PATH)
check = reloaded.predict(last_rows[feature_cols])
assert np.allclose(check, pred_correct)
print('reloaded pipeline reproduces the saved predictions exactly:', MODEL_PATH)


Full notes, with the physical-feature and spectral-feature material this
notebook does not cover: [`../notes.md`](notes.md).